# Phase 3: BM25 희소 검색

키워드 빈도 기반 BM25로 정확한 기술명 매칭을 테스트한다.
Dense가 놓치는 케이스(qa-001이 Java 검색에 상위 노출)를 BM25가 잡아내는지 비교.

In [1]:
from _bootstrap import setup_project_path

setup_project_path()

WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [3]:
from embedding_retrieval.search.bm25 import BM25Index
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEER_PROFILES

# capability / experience 각각 BM25 인덱스 구축
cap_bm25 = BM25Index()
cap_bm25.fit([(p.engineer_id, p.capability_text) for p in SAMPLE_ENGINEER_PROFILES])

exp_bm25 = BM25Index()
exp_bm25.fit([(p.engineer_id, p.experience_text) for p in SAMPLE_ENGINEER_PROFILES])

print(f"capability BM25: {cap_bm25.doc_count}문서, vocab {cap_bm25.vocab_size}개")
print(f"experience BM25: {exp_bm25.doc_count}문서, vocab {exp_bm25.vocab_size}개")

capability BM25: 9문서, vocab 54개
experience BM25: 9문서, vocab 223개


In [4]:
# Dense에서 qa-001이 0.89로 나왔던 문제 케이스
print("=== BM25 capability 검색: 'Java Spring' ===")
results = cap_bm25.search("Java Spring", top_k=9)
for doc_id, score in results:
    print(f"  {doc_id:8s}  bm25: {score:.4f}")

print()
print("→ qa-001(Selenium/JMeter)은 Java 키워드가 없으므로 점수 0 → 목록에 안 나옴")

=== BM25 capability 검색: 'Java Spring' ===
  eng-002   bm25: 3.1960
  eng-001   bm25: 3.0422

→ qa-001(Selenium/JMeter)은 Java 키워드가 없으므로 점수 0 → 목록에 안 나옴


In [7]:
# React 검색 — Dense에선 Vue.js도 비슷하게 나올 수 있지만 BM25는 정확히 구분
print("=== BM25 capability 검색: 'React' ===")
results = cap_bm25.search("React", top_k=9)
for doc_id, score in results:
    print(f"  {doc_id:8s}  bm25: {score:.4f}")

print()
print("→ React가 capability_text에 있는 eng-003, eng-004만 매칭")

=== BM25 capability 검색: 'React' ===
  eng-003   bm25: 1.4815
  eng-004   bm25: 1.4815

→ React가 capability_text에 있는 eng-003, eng-004만 매칭


In [8]:
# experience 검색: 도메인 키워드
print("=== BM25 experience 검색: '제조업 ERP' ===")
results = exp_bm25.search("제조업 ERP", top_k=9)
for doc_id, score in results:
    print(f"  {doc_id:8s}  bm25: {score:.4f}")

print()

print("=== BM25 experience 검색: '현대차' ===")
results = exp_bm25.search("현대차", top_k=9)
for doc_id, score in results:
    print(f"  {doc_id:8s}  bm25: {score:.4f}")

=== BM25 experience 검색: '제조업 ERP' ===
  eng-001   bm25: 5.3976

=== BM25 experience 검색: '현대차' ===
  eng-002   bm25: 1.9507


In [9]:
# BM25의 한계: '서버 개발자'로 검색하면 '백엔드'를 못 찾음
print("=== BM25 experience 검색: '서버 개발자' ===")
results = exp_bm25.search("서버 개발자", top_k=9)
for doc_id, score in results:
    print(f"  {doc_id:8s}  bm25: {score:.4f}")

print()
print("→ '백엔드 개발자'와 '서버 개발자'는 같은 의미지만 BM25는 키워드가 달라 매칭 안 됨")
print("→ 이걸 Dense가 보완 → 하이브리드의 필요성")

=== BM25 experience 검색: '서버 개발자' ===
  eng-005   bm25: 2.7170

→ '백엔드 개발자'와 '서버 개발자'는 같은 의미지만 BM25는 키워드가 달라 매칭 안 됨
→ 이걸 Dense가 보완 → 하이브리드의 필요성
